# Chakravyuh — Analyzer GRPO Training (Colab Pro A100 recommended)

**Meta PyTorch OpenEnv Hackathon 2026** — trains a LoRA adapter on **Qwen2.5-7B-Instruct** bf16.

- Repo: https://github.com/UjjwalPardeshi/Chakravyuh
- Training corpus: **408 examples** (327 scam / 81 benign), schema-disjoint from `chakravyuh-bench-v0` test set
- Scripted baseline (what LoRA must clear): **F1 = 0.903, novel det = 76.5%**

**Recommended: Colab Pro A100 40GB** — clean bf16 training, best model quality.

**Auto-selected config by GPU:**

| GPU | VRAM | Model | Precision | Temp | ~Time 408ep |
|---|---|---|---|---|---|
| **A100 / H100** | ≥30 GB | Qwen2.5-7B | bf16 | 1.1 | ~3-4 h |
| L4 / A10 / V100 | 20-30 GB | Qwen2.5-3B | 4-bit | 1.5 | ~6-8 h |
| T4 | <20 GB | Qwen2.5-1.5B | 4-bit | 1.5 | ~10-12 h |

**bf16 7B is the quality path** — the model is naturally diverse at temp=1.1 with no coherence cliff. Smaller 4-bit models need aggressive sampling (temp=1.5) and often produce partial gibberish.

**Before pressing Run All:**
1. `Runtime → Change runtime type → A100 GPU` (Colab Pro)
2. Connect to Drive when prompted (checkpoints persist there)
3. Keep the browser tab active (see Cell 1 for keep-alive trick)

## 0. Keep-alive (prevents Colab idle-disconnect)

Colab Free disconnects after ~90 min of tab inactivity. This cell prints a heartbeat every 30 seconds so the session stays alive when you close your laptop lid. Runs in the background — keep executing the remaining cells while this runs.

In [ ]:
# JavaScript keep-alive: clicks the 'connect' button every 60s.
# Open browser console (F12) and paste this if Colab disconnects during training:
#   function ClickConnect(){ document.querySelector('colab-toolbar-button#connect').click(); }
#   setInterval(ClickConnect, 60000)

from IPython.display import display, Javascript
display(Javascript('''
function KeepAlive() {
  const btn = document.querySelector('colab-toolbar-button#connect');
  if (btn) btn.click();
  const runBtn = document.querySelector('colab-connect-button');
  if (runBtn) runBtn.click();
}
setInterval(KeepAlive, 60000);
console.log('Colab keep-alive active');
'''))
print('Keep-alive installed. Session will resist idle-disconnect.')

## 1. Mount Google Drive (for checkpoint persistence)

Checkpoints save to Drive every N steps, so if Colab disconnects the next run resumes from the last checkpoint.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/chakravyuh'
CHECKPOINT_DIR = f'{DRIVE_ROOT}/analyzer_lora'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print('Checkpoint dir:', CHECKPOINT_DIR)

# Detect if a previous run left checkpoints worth resuming from.
RESUME = os.path.isdir(CHECKPOINT_DIR) and any(
    name.startswith('checkpoint-') for name in os.listdir(CHECKPOINT_DIR)
)
print('Resume from existing checkpoint?', RESUME)
if RESUME:
    print('Checkpoints found:', [n for n in os.listdir(CHECKPOINT_DIR) if n.startswith('checkpoint-')])

## 1b. (Optional) Load GROQ_API_KEY from Colab Secrets

The training reward has a +0.4 explanation-quality component judged by Llama-3.3-70B via Groq's free tier. Without a key, the reward falls back to a constant mock score — training still works, just with slightly less signal.

**To enable:** left sidebar → 🔑 Secrets → add `GROQ_API_KEY` (from [console.groq.com](https://console.groq.com)) → toggle "Notebook access" ON. Then run the cell below.

In [ ]:
import os

# Try Colab Secrets first (most users). Falls back silently if not set.
try:
    from google.colab import userdata  # type: ignore[import-not-found]
    _groq = userdata.get('GROQ_API_KEY')
    if _groq:
        os.environ['GROQ_API_KEY'] = _groq
        print('[OK] GROQ_API_KEY loaded from Colab Secrets — explanation judge will use Llama-3.3-70B.')
    else:
        print('[SKIP] No GROQ_API_KEY in Colab Secrets — reward will use mock judge (training still works).')
except Exception as e:
    print(f'[SKIP] Could not access Colab Secrets ({e}) — reward will use mock judge.')

# Quick sanity: print whether the env var is visible to subprocesses.
print('GROQ_API_KEY set in env:', bool(os.environ.get('GROQ_API_KEY')))

## 2. Install dependencies + clone repo

First run: 4-6 minutes (~2 GB of downloads). Subsequent runs are fast (already cached).

In [ ]:
# Core training stack. Pinned upper bounds — trl>=0.15 and transformers>=5.0 have breaking API changes.
#
# Unsloth is INTENTIONALLY skipped: unsloth>=2025.2 forcibly upgrades torch to 2.10+ with CUDA 13,
# which breaks bitsandbytes (built for CUDA 12) with "libnvJitLink.so.13: cannot open shared object".
# Pure 4-bit PEFT fallback in grpo_analyzer.py fits Qwen2.5-3B on T4 without the Unsloth 2x speedup.
#
# torchvision/torchaudio MUST be pinned alongside torch, else Colab's preinstalled
# torchvision 0.25+cu128 keeps its C++ extension built against torch 2.10 and
# explodes with "operator torchvision::nms does not exist" when transformers imports it.
#
# accelerate MUST be >=1.0: transformers 4.50+ calls Accelerator.unwrap_model(keep_torch_compile=False)
# which only exists in accelerate>=1.0. Older accelerate raises TypeError during trainer.train().
!pip install -q --upgrade \
    'torch>=2.5,<2.6' 'torchvision>=0.20,<0.21' 'torchaudio>=2.5,<2.6' \
    'transformers>=4.45,<5.0' 'trl>=0.12,<0.15' \
    'peft>=0.13,<0.15' 'accelerate>=1.0,<2.0' 'bitsandbytes>=0.44,<0.46' 'datasets>=2.19,<3.0'
# Project deps.
!pip install -q 'pydantic>=2.6' 'numpy>=1.26' 'jsonlines>=4.0' 'tqdm>=4.66'

# Verify pinned versions landed correctly before continuing.
import importlib
_versions = {m: importlib.import_module(m).__version__ for m in ('torch', 'trl', 'peft', 'transformers', 'accelerate', 'bitsandbytes')}
print('Installed:', _versions)

# Hard assertions — fail loudly if pip let something bad through.
assert _versions['torch'].startswith('2.5.'), f"Bad torch: {_versions['torch']}. Need 2.5.x."

_trl_major_minor = tuple(int(x) for x in _versions['trl'].split('.')[:2])
assert (0, 12) <= _trl_major_minor < (0, 15), f"Bad trl: {_versions['trl']}. Need 0.12-0.14."

_acc_major = int(_versions['accelerate'].split('.')[0])
assert _acc_major >= 1, f"Bad accelerate: {_versions['accelerate']}. Need >=1.0 (transformers 4.50+ requires keep_torch_compile kwarg)."

# Smoke-test torch / torchvision ABI match.
try:
    import torchvision, torchvision.ops  # noqa: F401
    print('[OK] torchvision ABI compatible with torch.')
except Exception as e:
    raise RuntimeError(f"torchvision import failed: {e}. Runtime → Disconnect and delete runtime, then Run All again.")

# Smoke-test bitsandbytes — we need it for 4-bit loading in the PEFT fallback path.
import bitsandbytes as bnb
try:
    _ = bnb.nn.Linear4bit
    print(f'[OK] bitsandbytes {bnb.__version__} loaded and Linear4bit available.')
except Exception as e:
    raise RuntimeError(f"bitsandbytes broken: {e}. Runtime → Disconnect and delete runtime, then Run All again.")

print('[OK] Versions compatible with GRPO training code (pure PEFT + 4-bit path).')

In [ ]:
# Clone OR pull latest. This is important — always use freshest GitHub code.
REPO_URL = 'https://github.com/UjjwalPardeshi/Chakravyuh.git'
REPO_DIR = '/content/Chakravyuh'

if os.path.exists(REPO_DIR):
    print('Repo exists — pulling latest...')
    !cd {REPO_DIR} && git fetch origin && git reset --hard origin/main && git clean -fd
else:
    print('Cloning repo...')
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print('Working dir:', os.getcwd())
!git log -1 --oneline

# Install our package with --no-deps so pip can't re-resolve torch/torchvision/transformers
# (which would silently break the ABI pins from Cell 8). All runtime deps were
# already installed in Cell 8.
!pip install -q -e . --no-deps

# Re-verify torch/torchvision ABI survived the editable install.
try:
    import torchvision, torchvision.ops  # noqa: F401
    print('[OK] torchvision ABI still compatible after -e .')
except Exception as e:
    raise RuntimeError(
        f'torchvision ABI broke after pip install -e .: {e}\n'
        'Runtime → Disconnect and delete runtime, then Run All again.'
    )
print('Setup complete.')

In [ ]:
# Sanity — confirm GPU + versions.
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM: {vram:.1f} GB')
    if vram < 14:
        print('[WARN] VRAM < 14 GB — may need to drop to Qwen2.5-1.5B-Instruct')
else:
    raise RuntimeError('No GPU! Go to Runtime → Change runtime type → T4 GPU')

import trl, peft, transformers
print(f'trl:{trl.__version__} | peft:{peft.__version__} | transformers:{transformers.__version__}')

## 3. Sanity check — build dataset + reward function

Verifies training pipeline loads correctly without touching the GPU. Expect: **408 examples (327 scam / 81 benign)**.

In [ ]:
from training.grpo_analyzer import build_training_examples, compute_reward

examples = build_training_examples()
n_scam = sum(1 for e in examples if e.is_scam)
n_benign = len(examples) - n_scam
print(f'Built {len(examples)} training examples ({n_scam} scam / {n_benign} benign)')
assert len(examples) >= 300, 'Training corpus unexpectedly small — check repo was pulled correctly'

# Reward function spot-check.
ex = next(e for e in examples if e.is_scam)
good = '{"score": 0.95, "signals": ["urgency", "info_request"], "explanation": "OTP + urgency = scam"}'
bad  = '{"score": 0.1,  "signals": [],                          "explanation": "Looks fine to me"}'
print(f'GOOD completion reward: {compute_reward(good, ex).total:+.3f}  (expect ~+1.3)')
print(f'BAD  completion reward: {compute_reward(bad, ex).total:+.3f}   (expect ~-0.2)')

### 3b. Smoke test — run 10 episodes first to verify gradients are flowing

This catches the `grad_norm=0` / `reward_std=0` failure mode early, before committing to a 4-6 hour run. Takes ~10-15 min on T4, ~5 min on A100. **If you see `grad_norm > 0.1` in the training log, you're good to cancel this and run the full training below.**

In [ ]:
# Smoke test: run 10 episodes, watch for grad_norm > 0.
# Outputs to a TEMP dir so it doesn't pollute the real checkpoint location.
import torch, subprocess, sys, os

_vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
if _vram_gb >= 30:
    _smoke_model = 'Qwen/Qwen2.5-7B-Instruct'
elif _vram_gb >= 20:
    _smoke_model = 'Qwen/Qwen2.5-3B-Instruct'
else:
    _smoke_model = 'Qwen/Qwen2.5-1.5B-Instruct'

_smoke_out = '/tmp/chakravyuh_smoke'
os.makedirs(_smoke_out, exist_ok=True)

smoke_cmd = [
    sys.executable, '-u', '-m', 'training.grpo_analyzer',
    '--model', _smoke_model,
    '--episodes', '10',
    '--output', _smoke_out,
    '--lora-rank', '32', '--lora-alpha', '64',
    '--learning-rate', '5e-5',
    '--batch-size', '1', '--grad-accum', '2',
    '--num-generations', '4',
    '--max-completion-length', '128',
    '--no-unsloth', '--no-wandb',
]
print('Smoke-test:', ' '.join(smoke_cmd))
print('---')
env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'
env['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
proc = subprocess.Popen(smoke_cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1, text=True, env=env)
for line in proc.stdout:
    print(line, end=''); sys.stdout.flush()
ret = proc.wait()
if ret == 0:
    print('\n[SMOKE OK] Pipeline runs end-to-end. Check the loss log above:')
    print('  - grad_norm > 0.01  → gradients flowing, safe to run full training')
    print('  - grad_norm ≈ 0     → stop, investigate (likely reward_std=0 from identical generations)')
else:
    print(f'\n[SMOKE FAILED] exit={ret}. Full training would also fail — fix before proceeding.')

## 4. Train the LoRA (auto-configured by GPU)

Training corpus size after the 04-22 expansion: **619 examples** (403 scam / 216 benign, incl. 30 hard negatives and 76 novel scam patterns across 6 new vectors).

`EPISODES = 619` uses the full corpus. Lower it if you want a quick smoke run:

| EPISODES | Time on A100 (7B bf16) | Time on L4 (3B 4-bit) | Use case |
|---|---|---|---|
| **50** | ~25 min | ~60-90 min | Quick smoke |
| **200** | ~1-1.5 h | ~4-6 h | Short overnight |
| **619** (full) | **~2.5-3 h** | ~9-12 h | **Recommended — best quality** |

The cell auto-selects:
- **A100 / H100 (≥30 GB)** → Qwen2.5-7B **bf16**, rank=64, batch=2×grad_accum=2, **temp=0.8** (best quality)
- **L4 / A10 (20-30 GB)** → Qwen2.5-3B **4-bit**, rank=32, batch=1×grad_accum=4, **temp=1.0**
- **T4 (<20 GB)** → Qwen2.5-1.5B **4-bit**, rank=32, batch=1×grad_accum=4, **temp=1.0**

**Expected telemetry on A100:**
- `grad_norm` 0.5-3 (gradients flowing)
- `reward_std` 0.3-0.8 (generations meaningfully differ)
- `reward` climbs from ~0.3 → ~1.2 over 310 steps
- `First prompt: ... N Qwen special-token markers (expect ≥5)` — if N<5, chat template is broken, stop and debug

In [ ]:
# ------- AUTO-CONFIGURE BASED ON GPU -------
# Detects VRAM and picks model + hyperparams that match.
#   A100 40/80 GB, H100  → Qwen2.5-7B bf16, rank=64, batch=2, temp=0.8 (best quality)
#   L4 / A10 22-24 GB    → Qwen2.5-3B 4-bit, rank=32, batch=1, temp=1.0
#   T4 16 GB              → Qwen2.5-1.5B 4-bit, rank=32, batch=1, temp=1.0
import torch

_vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'Detected GPU: {torch.cuda.get_device_name(0)} ({_vram_gb:.1f} GB VRAM)')

if _vram_gb >= 30:
    # A100 / H100 path — bf16, bigger LoRA, slightly larger effective batch.
    MODEL_NAME = 'Qwen/Qwen2.5-7B-Instruct'
    NUM_GENERATIONS = 4
    LORA_RANK = 64              # 7B has capacity to exploit r=64
    BATCH_SIZE = 2              # A100 40GB fits 2 comfortably with bf16 7B + LoRA
    GRAD_ACCUM = 2              # effective batch = 4
    # temp/top_p/top_k left as None → grpo_analyzer auto-picks 0.8/0.9/50 (bf16)
    TEMPERATURE = None
    print('→ Qwen2.5-7B bf16 | rank=64 | batch=2×2 | temp=auto(0.8)')
elif _vram_gb >= 20:
    MODEL_NAME = 'Qwen/Qwen2.5-3B-Instruct'
    NUM_GENERATIONS = 4
    LORA_RANK = 32
    BATCH_SIZE = 1
    GRAD_ACCUM = 4              # effective batch = 4
    TEMPERATURE = None          # auto-picks 1.0 (4-bit)
    print('→ Qwen2.5-3B 4-bit | rank=32 | batch=1×4 | temp=auto(1.0)')
else:
    MODEL_NAME = 'Qwen/Qwen2.5-1.5B-Instruct'
    NUM_GENERATIONS = 4
    LORA_RANK = 32
    BATCH_SIZE = 1
    GRAD_ACCUM = 4
    TEMPERATURE = None
    print('→ Qwen2.5-1.5B 4-bit | rank=32 | batch=1×4 | temp=auto(1.0)')

# Full corpus after the 04-22 expansion: 619 examples (403 scam / 216 benign).
# On A100, ~28 sec/step × 310 steps (619 / batch=2) ≈ 2.5 hours.
EPISODES = 619
LEARNING_RATE = 5e-5        # LoRA needs higher LR than full-ft's 1e-5
MAX_COMPLETION_LEN = 128    # JSON target is ~50 tokens, 128 is plenty

import subprocess, sys, os

cmd = [
    sys.executable, '-u', '-m', 'training.grpo_analyzer',
    '--model', MODEL_NAME,
    '--episodes', str(EPISODES),
    '--output', CHECKPOINT_DIR,
    '--lora-rank', str(LORA_RANK),
    '--lora-alpha', str(LORA_RANK * 2),
    '--learning-rate', str(LEARNING_RATE),
    '--batch-size', str(BATCH_SIZE),
    '--grad-accum', str(GRAD_ACCUM),
    '--num-generations', str(NUM_GENERATIONS),
    '--max-completion-length', str(MAX_COMPLETION_LEN),
    '--no-unsloth',           # Unsloth's torch-upgrade aggression breaks bitsandbytes
    '--no-wandb',
]
if TEMPERATURE is not None:
    cmd.extend(['--temperature', str(TEMPERATURE)])
if RESUME:
    cmd.append('--resume')
    print('[RESUMING from last checkpoint]')

print('Launching:', ' '.join(cmd))
print('---')

env = os.environ.copy()
env.setdefault('PYTHONUNBUFFERED', '1')
env.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    bufsize=1,
    text=True,
    env=env,
)
assert proc.stdout is not None
try:
    for line in proc.stdout:
        print(line, end='')
        sys.stdout.flush()
finally:
    ret = proc.wait()

if ret != 0:
    print(f'\n[FAILED] Training returned exit code {ret}. Full traceback is above.')
    print('Common fixes:')
    print('  - OOM on A100: lower BATCH_SIZE to 1, GRAD_ACCUM to 4')
    print('  - OOM on L4/T4: change MODEL_NAME to a smaller Qwen variant')
    print('  - Still OOM: lower NUM_GENERATIONS to 2 (GRPO quality suffers)')
    print('  - bitsandbytes errors: Runtime → Disconnect and delete runtime → Run all from Cell 8')
    print('  - Drive full: prune old checkpoints under', CHECKPOINT_DIR)
else:
    print('\n[SUCCESS] Training completed. LoRA adapter saved to', CHECKPOINT_DIR)

## 5. Evaluate the trained adapter on chakravyuh-bench-v0 (175 scenarios)

This cell takes ~15-30 min on T4 (175 inferences × ~5-10s each).

Baseline to beat: **ScriptedAnalyzer F1 = 0.903, detection = 84.0%, FPR = 9.7%, novel = 76.5%**

In [ ]:
from chakravyuh_env.agents.llm_analyzer import LLMAnalyzer
from eval.mode_c_real_cases import (
    load_dataset, run_eval, aggregate,
    per_category_breakdown, per_difficulty_breakdown,
    DEFAULT_DATASET,
)

print(f'Loading LoRA adapter from {CHECKPOINT_DIR}')
print(f'Base model: {MODEL_NAME}')

# Match training precision: bf16 on A100/L4, 4-bit on T4. Falls back gracefully.
_load_in_4bit = _vram_gb < 30

analyzer = LLMAnalyzer(
    model_name=MODEL_NAME,
    lora_path=CHECKPOINT_DIR,
    use_unsloth=False,              # matches training — keeps load paths consistent
    load_in_4bit=_load_in_4bit,
)
analyzer.load()
print('Model loaded. Starting evaluation...')

data = load_dataset(DEFAULT_DATASET)
results = run_eval(analyzer, data, threshold=0.5)
metrics = aggregate(results)

print()
print('=== LoRA vs scripted baseline (F1=0.903) ===')
print(f'  Detection: {metrics.detection_rate:.1%}  (scripted: 84.0%)')
print(f'  Precision: {metrics.precision:.1%}  (scripted: 97.6%)')
print(f'  FPR:       {metrics.false_positive_rate:.1%}  (scripted: 9.7%)')
print(f'  F1:        {metrics.f1:.3f}  (scripted: 0.903)')

by_diff = per_difficulty_breakdown(results)
print()
for diff, m in sorted(by_diff.items()):
    print(f'  {diff:10s}: det={m.detection_rate:.1%}  (n={m.n})')

## 6. Save eval results to Drive

Writes `eval_results.json` next to the LoRA adapter, so you can reference exact numbers later.

In [ ]:
import json
from datetime import datetime

lora_eval = {
    'trained_on': datetime.now().isoformat(),
    'model': MODEL_NAME,
    'episodes': EPISODES,
    'lora_path': CHECKPOINT_DIR,
    'n_eval': metrics.n,
    'detection_rate': metrics.detection_rate,
    'false_positive_rate': metrics.false_positive_rate,
    'precision': metrics.precision,
    'recall': metrics.recall,
    'f1': metrics.f1,
    'accuracy': metrics.accuracy,
    'by_category': {k: v.__dict__ for k, v in per_category_breakdown(results).items()},
    'by_difficulty': {k: v.__dict__ for k, v in per_difficulty_breakdown(results).items()},
}
out_path = f'{CHECKPOINT_DIR}/eval_results.json'
with open(out_path, 'w') as f:
    json.dump(lora_eval, f, indent=2)
print('Saved:', out_path)
print('\nLoRA adapter ready for:')
print(' - Download from Drive UI (right-click analyzer_lora folder)')
print(' - Or: huggingface-cli upload chakravyuh/analyzer-lora $CHECKPOINT_DIR')

## Next steps

1. **LoRA beat scripted baseline (F1 > 0.903)** → ship it. Download the adapter folder from Drive, commit to `checkpoints/` in your repo, update `baselines.json` with the new numbers.
2. **LoRA roughly tied scripted (F1 ~0.88-0.92)** → bump `EPISODES` to 408 and re-run (resumable — picks up where you left off). Also try `lora-rank 32` for more capacity.
3. **LoRA underperformed scripted (F1 < 0.85)** → likely reward-weight issue. Inspect `compute_reward` in `training/grpo_analyzer.py`, especially the false-positive penalty weight.
4. **OOM / session died** → rerun this notebook. Cell 1 auto-detects checkpoints and resumes.

Chakravyuh · Meta PyTorch OpenEnv Hackathon · April 2026